# Phase 7: FFN Neuron Specialization — The Emotional Neurons of BERT

The feed-forward network (FFN) in each BERT encoder layer contains an **intermediate layer** with **3072 neurons** (after GELU activation). Across 12 layers, this amounts to **36,864 neurons** total.

In this notebook we analyze which of these neurons activate differentially for each of the 28 emotions in GoEmotions. This is the most **granular level of analysis** in the project — individual neuron behavior rather than layer-level or component-level metrics.

**Why this matters for SVD compression:** When SVD truncates the weight matrix of the FFN intermediate layer, it removes directions in the 3072-dimensional neuron space. If those removed directions align with the activation patterns of emotion-specific neurons, the corresponding emotions will be disproportionately damaged. This notebook establishes that connection explicitly.

**Sections:**
1. Extract FFN intermediate activations (post-GELU) via forward hooks
2. Compute neuron selectivity scores (effect size per neuron per emotion)
3. Catalog the most emotion-selective neurons
4. Population analysis — how many specialized neurons per emotion/layer
5. Connect neuron specialization to SVD truncation
6. Cluster emotions by shared neuron profiles
7. Save results

In [ ]:
import sys
import os

# ---------- Colab / local compatibility ----------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')

# ---------- Imports ----------
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist
from sklearn.decomposition import PCA
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

from src.data import load_goemotions
from src.data.dataset import NUM_LABELS, EMOTION_NAMES
from src.compression import (
    apply_svd_compression,
    compute_singular_value_energy,
    get_target_layer_names,
    filter_layer_names,
    count_parameters,
)
from src.utils import compute_metrics

sns.set_style('whitegrid')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Neurons per layer: 3072')
print(f'Total FFN neurons: {12 * 3072:,}')

In [ ]:
# ---------- Load model and data ----------
model_path = os.path.join(PROJECT_ROOT, 'results', 'bert-goemotions-final')

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=NUM_LABELS,
    problem_type='multi_label_classification',
)
model.eval()
model.to(device)

params = count_parameters(model)
print(f'Parameters: {params["total"]:,}')

_, _, test_ds, emotion_names, data_collator = load_goemotions()
print(f'Test set: {len(test_ds)} examples')

results_dir = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(results_dir, exist_ok=True)

## 1. Extract FFN Intermediate Activations

We use **forward hooks** on `BertIntermediate` modules to capture the post-GELU activations
(shape: batch x seq_len x 3072). We **mean-pool** over sequence positions to get a single
3072-dimensional activation vector per example, capturing the overall firing pattern of each neuron.

In [ ]:
@torch.no_grad()
def extract_ffn_activations(model, dataset, data_collator, batch_size=64):
    """Extract FFN intermediate activations (post-GELU) for all 12 layers.

    Returns:
        activations: dict layer_idx -> numpy array (N, 3072)
        labels: numpy array (N, 28)
    """
    dev = next(model.parameters()).device
    n_layers = model.config.num_hidden_layers

    # Use hooks to capture intermediate activations
    activations = {l: [] for l in range(n_layers)}
    hooks = []

    for layer_idx in range(n_layers):
        def hook_fn(module, input, output, l=layer_idx):
            # BertIntermediate output: (batch, seq, 3072) after GELU
            # Mean-pool over sequence positions (captures overall neuron activation)
            mean_act = output.mean(dim=1).cpu()  # (batch, 3072)
            activations[l].append(mean_act)

        h = model.bert.encoder.layer[layer_idx].intermediate.register_forward_hook(hook_fn)
        hooks.append(h)

    all_labels = []
    n_batches = (len(dataset) + batch_size - 1) // batch_size
    for i in range(0, len(dataset), batch_size):
        batch = data_collator([dataset[j] for j in range(i, min(i + batch_size, len(dataset)))])
        model(input_ids=batch['input_ids'].to(dev),
              attention_mask=batch['attention_mask'].to(dev))
        all_labels.append(batch['labels'].numpy())

        if (i // batch_size + 1) % 20 == 0:
            print(f'  Batch {i // batch_size + 1}/{n_batches}')

    for h in hooks:
        h.remove()

    for l in range(n_layers):
        activations[l] = torch.cat(activations[l], dim=0).numpy()
    labels = np.concatenate(all_labels, axis=0)

    return activations, labels

In [ ]:
print('Extracting FFN intermediate activations from test set...')
activations, labels = extract_ffn_activations(model, test_ds, data_collator)

n_samples = labels.shape[0]
n_layers = len(activations)
n_neurons = activations[0].shape[1]

print(f'\nExtraction complete:')
print(f'  Samples: {n_samples}')
print(f'  Layers: {n_layers}')
print(f'  Neurons per layer: {n_neurons}')
print(f'  Activation shape per layer: {activations[0].shape}')
print(f'  Labels shape: {labels.shape}')

# Memory usage
mem_bytes = sum(activations[l].nbytes for l in range(n_layers)) + labels.nbytes
print(f'  Total memory: {mem_bytes / 1024**2:.1f} MB')

# Quick stats
print(f'\nActivation statistics (layer 0):')
print(f'  Mean: {activations[0].mean():.4f}')
print(f'  Std:  {activations[0].std():.4f}')
print(f'  Min:  {activations[0].min():.4f}')
print(f'  Max:  {activations[0].max():.4f}')

# Label distribution
print(f'\nEmotion prevalence in test set:')
for emo_idx in range(NUM_LABELS):
    count = int(labels[:, emo_idx].sum())
    print(f'  {EMOTION_NAMES[emo_idx]:>15s}: {count:>4d} ({count/n_samples*100:5.1f}%)')

## 2. Neuron Selectivity Scores

For each neuron in each layer, we compute a **selectivity score** for each emotion using a
standardized effect size (analogous to Cohen's d):

$$\text{selectivity}(n, e) = \frac{\bar{a}_{n}^{(e+)} - \bar{a}_{n}^{(e-)}}{s_{\text{pooled}}}$$

where $\bar{a}_{n}^{(e+)}$ is the mean activation of neuron $n$ when emotion $e$ is present,
$\bar{a}_{n}^{(e-)}$ when absent, and $s_{\text{pooled}}$ is the pooled standard deviation.

A selectivity score > 2.0 indicates that the neuron fires substantially more (or less) when
a given emotion is present — a strong "emotion detector".

In [ ]:
def compute_neuron_selectivity(activations, labels):
    """Compute selectivity score for each neuron for each emotion.

    Selectivity = (mean_activation_when_emotion_present - mean_activation_when_absent)
                  / pooled_std

    Returns: dict layer_idx -> numpy array (3072, 28) of selectivity scores
    """
    n_layers = len(activations)
    n_emotions = labels.shape[1]
    selectivity = {}

    for layer_idx in range(n_layers):
        act = activations[layer_idx]  # (N, 3072)
        sel = np.zeros((act.shape[1], n_emotions))

        for emo_idx in range(n_emotions):
            mask = labels[:, emo_idx] > 0.5
            if mask.sum() < 5 or (~mask).sum() < 5:
                continue

            mean_pos = act[mask].mean(axis=0)
            mean_neg = act[~mask].mean(axis=0)
            std_pos = act[mask].std(axis=0) + 1e-8
            std_neg = act[~mask].std(axis=0) + 1e-8
            n_pos, n_neg = mask.sum(), (~mask).sum()
            pooled_std = np.sqrt(((n_pos - 1) * std_pos**2 + (n_neg - 1) * std_neg**2)
                                 / (n_pos + n_neg - 2)) + 1e-8

            sel[:, emo_idx] = (mean_pos - mean_neg) / pooled_std

        selectivity[layer_idx] = sel
        print(f'  Layer {layer_idx:2d}: max selectivity = {np.abs(sel).max():.3f}, '
              f'mean |sel| = {np.abs(sel).mean():.4f}')

    return selectivity


print('Computing neuron selectivity scores...')
selectivity = compute_neuron_selectivity(activations, labels)

# Global stats
all_sel = np.concatenate([selectivity[l].ravel() for l in range(n_layers)])
print(f'\nGlobal selectivity statistics:')
print(f'  Total neuron-emotion pairs: {len(all_sel):,}')
print(f'  Mean |selectivity|: {np.abs(all_sel).mean():.4f}')
print(f'  Max |selectivity|:  {np.abs(all_sel).max():.4f}')
print(f'  Pairs with |sel| > 1.0: {(np.abs(all_sel) > 1.0).sum():,}')
print(f'  Pairs with |sel| > 2.0: {(np.abs(all_sel) > 2.0).sum():,}')
print(f'  Pairs with |sel| > 3.0: {(np.abs(all_sel) > 3.0).sum():,}')

## 3. The Most Emotional Neurons

For each emotion, we identify the **top 10 most selective neurons** across all layers.
These are the neurons that function as "emotion detectors" — they fire significantly
more (or less) when a particular emotion is present in the text.

In [ ]:
# Build catalog of top neurons per emotion
TOP_K = 10
catalog_rows = []

for emo_idx in range(NUM_LABELS):
    # Collect all (layer, neuron, selectivity) tuples for this emotion
    all_scores = []
    for layer_idx in range(n_layers):
        sel = selectivity[layer_idx][:, emo_idx]  # (3072,)
        for neuron_idx in range(len(sel)):
            all_scores.append((layer_idx, neuron_idx, sel[neuron_idx]))

    # Sort by absolute selectivity
    all_scores.sort(key=lambda x: abs(x[2]), reverse=True)

    for rank, (layer_idx, neuron_idx, score) in enumerate(all_scores[:TOP_K]):
        catalog_rows.append({
            'emotion': EMOTION_NAMES[emo_idx],
            'emotion_idx': emo_idx,
            'rank': rank + 1,
            'layer': layer_idx,
            'neuron': neuron_idx,
            'selectivity': score,
            'abs_selectivity': abs(score),
            'direction': 'excitatory' if score > 0 else 'inhibitory',
        })

df_catalog = pd.DataFrame(catalog_rows)

print(f'Neuron catalog: {len(df_catalog)} entries ({NUM_LABELS} emotions x {TOP_K} neurons)\n')

# Show top 3 neurons per emotion
print(f'{"Emotion":>15s} | {"#1 (L,N)":>12s} sel   | {"#2 (L,N)":>12s} sel   | {"#3 (L,N)":>12s} sel')
print('-' * 85)
for emo_idx in range(NUM_LABELS):
    emo_rows = df_catalog[df_catalog['emotion_idx'] == emo_idx].head(3)
    parts = [f'{EMOTION_NAMES[emo_idx]:>15s} |']
    for _, row in emo_rows.iterrows():
        parts.append(f' ({row["layer"]:2d},{row["neuron"]:4d}) {row["selectivity"]:+.3f} |')
    print(''.join(parts))

In [ ]:
# Heatmap: max selectivity per emotion per layer
# Shape: 28 emotions x 12 layers
max_sel_matrix = np.zeros((NUM_LABELS, n_layers))
for layer_idx in range(n_layers):
    for emo_idx in range(NUM_LABELS):
        max_sel_matrix[emo_idx, layer_idx] = np.abs(selectivity[layer_idx][:, emo_idx]).max()

# Sort emotions by the layer where they have peak selectivity (then by magnitude)
peak_layer = np.argmax(max_sel_matrix, axis=1)
peak_value = np.max(max_sel_matrix, axis=1)
sort_order = np.lexsort((-peak_value, peak_layer))

sorted_matrix = max_sel_matrix[sort_order]
sorted_names = [EMOTION_NAMES[i] for i in sort_order]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    sorted_matrix, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=[str(l) for l in range(n_layers)],
    yticklabels=sorted_names,
    ax=ax, linewidths=0.5,
)
ax.set_xlabel('Encoder Layer', fontsize=12)
ax.set_ylabel('Emotion', fontsize=12)
ax.set_title('Peak Neuron Selectivity by Emotion and Layer\n'
             '(max |selectivity| across 3072 neurons per layer)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'neuron_selectivity_heatmap.png'),
            dpi=200, bbox_inches='tight')
plt.show()

print('\nEmotions sorted by peak layer:')
for i, idx in enumerate(sort_order):
    print(f'  {EMOTION_NAMES[idx]:>15s}: peak at layer {peak_layer[idx]}, '
          f'max sel = {peak_value[idx]:.3f}')

## 4. Neuron Population Analysis

We examine the **population-level** distribution of neuron specialization:
- How many neurons are "significantly selective" for each emotion?
- Which layers contain the most specialized neurons?
- What does the distribution of selectivity look like within each layer?

In [ ]:
# Count significantly selective neurons per emotion (|selectivity| > 2.0)
SELECTIVITY_THRESHOLD = 2.0

sig_counts = []
for emo_idx in range(NUM_LABELS):
    total_sig = 0
    per_layer = []
    for layer_idx in range(n_layers):
        n_sig = (np.abs(selectivity[layer_idx][:, emo_idx]) > SELECTIVITY_THRESHOLD).sum()
        total_sig += n_sig
        per_layer.append(n_sig)
    sig_counts.append({
        'emotion': EMOTION_NAMES[emo_idx],
        'total_significant': total_sig,
        **{f'layer_{l}': per_layer[l] for l in range(n_layers)},
    })

df_sig = pd.DataFrame(sig_counts).sort_values('total_significant', ascending=True)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(df_sig)))
bars = ax.barh(range(len(df_sig)), df_sig['total_significant'], color=colors)
ax.set_yticks(range(len(df_sig)))
ax.set_yticklabels(df_sig['emotion'])
ax.set_xlabel(f'Number of Specialized Neurons (|selectivity| > {SELECTIVITY_THRESHOLD})', fontsize=11)
ax.set_title(f'Neuron Specialization by Emotion\n'
             f'(across all 12 layers x 3072 neurons = {12*3072:,} total)',
             fontsize=13, fontweight='bold')

# Add count labels
for i, (_, row) in enumerate(df_sig.iterrows()):
    ax.text(row['total_significant'] + 5, i, str(int(row['total_significant'])),
            va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'neuron_specialization_per_emotion.png'),
            dpi=200, bbox_inches='tight')
plt.show()

# Summary
print(f'\nEmotions with FEWEST specialized neurons (most fragile under compression):')
for _, row in df_sig.head(5).iterrows():
    print(f'  {row["emotion"]:>15s}: {int(row["total_significant"]):4d} neurons')
print(f'\nEmotions with MOST specialized neurons (most robust):')
for _, row in df_sig.tail(5).iterrows():
    print(f'  {row["emotion"]:>15s}: {int(row["total_significant"]):4d} neurons')

In [ ]:
# Distribution of max selectivity (across emotions) per neuron, by layer
# For each neuron, what is the strongest emotion signal it carries?

fig, axes = plt.subplots(3, 4, figsize=(18, 12), sharey=True)

for layer_idx in range(n_layers):
    ax = axes[layer_idx // 4, layer_idx % 4]
    sel = selectivity[layer_idx]  # (3072, 28)
    max_sel_per_neuron = np.abs(sel).max(axis=1)  # (3072,)

    # Histogram
    ax.hist(max_sel_per_neuron, bins=60, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(x=SELECTIVITY_THRESHOLD, color='red', linestyle='--', linewidth=1.5,
               label=f'threshold = {SELECTIVITY_THRESHOLD}')

    n_specialized = (max_sel_per_neuron > SELECTIVITY_THRESHOLD).sum()
    pct = n_specialized / len(max_sel_per_neuron) * 100

    ax.set_title(f'Layer {layer_idx}\n{n_specialized} specialized ({pct:.1f}%)',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Max |selectivity|' if layer_idx >= 8 else '')
    if layer_idx % 4 == 0:
        ax.set_ylabel('Neuron count')

axes[0, 0].legend(fontsize=8)

fig.suptitle('Distribution of Neuron Specialization per Layer\n'
             '(max |selectivity| across 28 emotions for each of 3072 neurons)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'neuron_selectivity_distributions.png'),
            dpi=200, bbox_inches='tight')
plt.show()

# Summary: specialized neurons per layer
print('\nSpecialized neurons per layer (|sel| > 2.0):')
for layer_idx in range(n_layers):
    sel = selectivity[layer_idx]
    max_sel = np.abs(sel).max(axis=1)
    n_spec = (max_sel > SELECTIVITY_THRESHOLD).sum()
    print(f'  Layer {layer_idx:2d}: {n_spec:4d} / 3072 ({n_spec/3072*100:.1f}%)')

## 5. Connecting Neurons to SVD

This is the **key analysis** of the notebook. SVD removes the least important singular vectors
from the weight matrix. Do these removed directions correspond to emotion-specific neuron
activation patterns?

For each layer's FFN intermediate weight matrix $W$ (3072 x 768):
- Compute SVD: $W = U \Sigma V^T$
- The columns of $U$ define an orthonormal basis in the 3072-dim neuron space
- The **first** $k$ columns (kept by SVD) span the "important" directions
- The **remaining** columns (removed by SVD truncation) span the "discarded" directions

We project each emotion's selectivity vector onto both subspaces. If an emotion's selectivity
signal lives primarily in the removed subspace, SVD will destroy that emotion's representation.

In [ ]:
def analyze_svd_neuron_connection(model, selectivity, rank=128):
    """Check if SVD-removed dimensions align with emotion-specific neurons.

    For each layer, projects the emotion selectivity vector onto the kept
    vs removed subspaces of the SVD decomposition.

    Returns:
        results: dict (layer_idx, emo_idx) -> {'frac_in_removed', 'frac_in_kept'}
    """
    results = {}

    for layer_idx in range(12):
        # Get FFN intermediate weight
        W = model.bert.encoder.layer[layer_idx].intermediate.dense.weight.data.float()
        # W shape: (3072, 768) -- maps 768 -> 3072

        U, S, Vh = torch.linalg.svd(W, full_matrices=False)
        # U: (3072, 768), columns of U are neuron-space directions

        # Directions KEPT by SVD (first `rank` columns of U)
        U_kept = U[:, :rank].cpu().numpy()  # (3072, rank)
        # Directions REMOVED (remaining columns)
        U_removed = U[:, rank:].cpu().numpy()  # (3072, 768-rank)

        # For each emotion, check if its selective neurons are in the removed subspace
        sel = selectivity[layer_idx]  # (3072, 28)

        for emo_idx in range(NUM_LABELS):
            # Get the neuron selectivity vector for this emotion
            neuron_sel = sel[:, emo_idx]  # (3072,)

            # Project onto kept vs removed subspace
            proj_kept = np.linalg.norm(U_kept.T @ neuron_sel)
            proj_removed = np.linalg.norm(U_removed.T @ neuron_sel)
            total = proj_kept + proj_removed + 1e-8

            results[(layer_idx, emo_idx)] = {
                'frac_in_removed': proj_removed / total,
                'frac_in_kept': proj_kept / total,
            }

        print(f'  Layer {layer_idx:2d}: singular values range '
              f'[{S[-1].item():.4f}, {S[0].item():.4f}], '
              f'rank {rank} retains {(S[:rank]**2).sum() / (S**2).sum() * 100:.1f}% energy')

    return results


ANALYSIS_RANK = 128
print(f'Analyzing SVD-neuron connection at rank {ANALYSIS_RANK}...')
svd_neuron_results = analyze_svd_neuron_connection(model, selectivity, rank=ANALYSIS_RANK)

In [ ]:
# Heatmap: fraction of emotion signal in removed subspace
frac_removed_matrix = np.zeros((NUM_LABELS, n_layers))
for layer_idx in range(n_layers):
    for emo_idx in range(NUM_LABELS):
        frac_removed_matrix[emo_idx, layer_idx] = \
            svd_neuron_results[(layer_idx, emo_idx)]['frac_in_removed']

# Sort by mean fraction removed (most vulnerable at top)
mean_frac_removed = frac_removed_matrix.mean(axis=1)
sort_order_vuln = np.argsort(-mean_frac_removed)

sorted_frac = frac_removed_matrix[sort_order_vuln]
sorted_names_vuln = [EMOTION_NAMES[i] for i in sort_order_vuln]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    sorted_frac, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=[str(l) for l in range(n_layers)],
    yticklabels=sorted_names_vuln,
    ax=ax, linewidths=0.5,
    vmin=0, vmax=0.7,
)
ax.set_xlabel('Encoder Layer', fontsize=12)
ax.set_ylabel('Emotion (sorted by vulnerability)', fontsize=12)
ax.set_title(f'Fraction of Emotion Signal in SVD-Removed Subspace (rank={ANALYSIS_RANK})\n'
             'Higher = more signal destroyed by SVD truncation',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'svd_neuron_vulnerability_heatmap.png'),
            dpi=200, bbox_inches='tight')
plt.show()

print(f'\nMost vulnerable emotions (highest fraction of signal in removed subspace):')
for idx in sort_order_vuln[:5]:
    print(f'  {EMOTION_NAMES[idx]:>15s}: {mean_frac_removed[idx]:.3f} avg across layers')
print(f'\nLeast vulnerable emotions:')
for idx in sort_order_vuln[-5:]:
    print(f'  {EMOTION_NAMES[idx]:>15s}: {mean_frac_removed[idx]:.3f} avg across layers')

In [ ]:
# Validation: compare SVD vulnerability with actual F1 impact under compression
# Compress the model at the analysis rank and measure per-emotion F1

print(f'Compressing model at rank {ANALYSIS_RANK} (FFN intermediate only) for validation...')

all_layer_names = get_target_layer_names(model)
ffn_inter_names = filter_layer_names(all_layer_names, component='intermediate')
print(f'  Compressing {len(ffn_inter_names)} layers: {ffn_inter_names[0]} ... {ffn_inter_names[-1]}')

compressed_model = apply_svd_compression(model, rank=ANALYSIS_RANK, layer_names=ffn_inter_names)
compressed_model.to(device)

# Evaluate baseline and compressed per-emotion F1
eval_args = TrainingArguments(
    output_dir=os.path.join(PROJECT_ROOT, 'results', 'tmp_eval'),
    per_device_eval_batch_size=64,
    fp16=(device == 'cuda'),
    report_to='none',
)

trainer_base = Trainer(
    model=model, args=eval_args,
    compute_metrics=compute_metrics, data_collator=data_collator,
)
trainer_comp = Trainer(
    model=compressed_model, args=eval_args,
    compute_metrics=compute_metrics, data_collator=data_collator,
)

print('  Evaluating baseline...')
base_metrics = trainer_base.evaluate(test_ds)
print('  Evaluating compressed...')
comp_metrics = trainer_comp.evaluate(test_ds)

print(f'\n  Baseline F1 macro: {base_metrics["eval_f1_macro"]:.4f}')
print(f'  Compressed F1 macro: {comp_metrics["eval_f1_macro"]:.4f}')

# Extract per-emotion F1 scores
per_emotion_f1_drop = []
for emo_idx in range(NUM_LABELS):
    key = f'eval_f1_{EMOTION_NAMES[emo_idx]}'
    if key in base_metrics and key in comp_metrics:
        base_f1 = base_metrics[key]
        comp_f1 = comp_metrics[key]
        drop = base_f1 - comp_f1
        per_emotion_f1_drop.append(drop)
    else:
        per_emotion_f1_drop.append(np.nan)

per_emotion_f1_drop = np.array(per_emotion_f1_drop)

del compressed_model
if device == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# Scatter plot: SVD vulnerability vs actual F1 drop
valid_mask = ~np.isnan(per_emotion_f1_drop)
x = mean_frac_removed[valid_mask]
y = per_emotion_f1_drop[valid_mask]
valid_names = [EMOTION_NAMES[i] for i in range(NUM_LABELS) if valid_mask[i]]

# Spearman correlation
rho, p_value = spearmanr(x, y)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(x, y, s=80, c='steelblue', edgecolor='white', linewidth=0.5, zorder=3)

# Label each point
for i, name in enumerate(valid_names):
    ax.annotate(name, (x[i], y[i]), fontsize=7, ha='center', va='bottom',
                xytext=(0, 5), textcoords='offset points')

# Trend line
z = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, np.polyval(z, x_line), '--', color='red', alpha=0.7, linewidth=1.5)

ax.set_xlabel('Fraction of Emotion Signal in Removed Subspace\n(SVD vulnerability)', fontsize=11)
ax.set_ylabel(f'F1 Drop under Compression (rank={ANALYSIS_RANK})', fontsize=11)
ax.set_title(f'SVD Vulnerability vs Actual F1 Impact\n'
             f'Spearman rho = {rho:.3f}, p = {p_value:.4f}',
             fontsize=13, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'svd_vulnerability_vs_f1_drop.png'),
            dpi=200, bbox_inches='tight')
plt.show()

print(f'Spearman correlation: rho = {rho:.3f}, p = {p_value:.4f}')
if p_value < 0.05:
    print('=> Significant correlation: emotions whose neuron signal lies in the')
    print('   SVD-removed subspace indeed suffer more F1 degradation.')
else:
    print('=> Correlation not significant at p < 0.05.')
    print('   The relationship may be more complex (non-linear, layer-dependent, etc.).')

## 6. Neuron Clustering

We cluster emotions by their **neuron activation profiles**: for each emotion, we concatenate
its selectivity vector across all 12 layers (12 x 3072 = 36,864 dimensions), reduce to 50
dimensions with PCA, and perform hierarchical clustering.

Emotions that share the same specialized neurons should cluster together — revealing the
"emotional genealogy" of BERT's FFN representations.

In [ ]:
# Build emotion feature vectors from selectivity profiles
# For each emotion: concatenate selectivity across all layers -> 36864 dims
emotion_features = np.zeros((NUM_LABELS, n_layers * n_neurons))
for emo_idx in range(NUM_LABELS):
    for layer_idx in range(n_layers):
        start = layer_idx * n_neurons
        end = start + n_neurons
        emotion_features[emo_idx, start:end] = selectivity[layer_idx][:, emo_idx]

print(f'Emotion feature matrix: {emotion_features.shape}')

# PCA reduction to 50 dimensions
N_COMPONENTS = 50
pca = PCA(n_components=N_COMPONENTS, random_state=42)
emotion_features_pca = pca.fit_transform(emotion_features)
explained_var = pca.explained_variance_ratio_.sum()
print(f'PCA: {N_COMPONENTS} components explain {explained_var*100:.1f}% of variance')

# Hierarchical clustering
distances = pdist(emotion_features_pca, metric='cosine')
linkage_matrix = linkage(distances, method='ward')

# Dendrogram
fig, ax = plt.subplots(figsize=(14, 8))
dendrogram(
    linkage_matrix,
    labels=EMOTION_NAMES,
    leaf_rotation=45,
    leaf_font_size=10,
    ax=ax,
    color_threshold=0.7 * linkage_matrix[-1, 2],
)
ax.set_ylabel('Distance', fontsize=12)
ax.set_title('Emotional Genealogy: Hierarchical Clustering of Neuron Profiles\n'
             '(emotions sharing specialized neurons cluster together)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'neuron_emotion_dendrogram.png'),
            dpi=200, bbox_inches='tight')
plt.show()

# Show cluster assignments
n_clusters = 6
cluster_labels = fcluster(linkage_matrix, n_clusters, criterion='maxclust')

print(f'\nCluster assignments ({n_clusters} clusters):')
for c in range(1, n_clusters + 1):
    members = [EMOTION_NAMES[i] for i in range(NUM_LABELS) if cluster_labels[i] == c]
    print(f'  Cluster {c}: {members}')

In [ ]:
# Similarity matrix of emotions based on shared neuron profiles
from scipy.spatial.distance import squareform

sim_matrix = 1 - squareform(distances)

# Reorder by dendrogram leaves
from scipy.cluster.hierarchy import leaves_list
leaf_order = leaves_list(linkage_matrix)
sim_reordered = sim_matrix[np.ix_(leaf_order, leaf_order)]
names_reordered = [EMOTION_NAMES[i] for i in leaf_order]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    sim_reordered, annot=False, cmap='RdBu_r',
    xticklabels=names_reordered,
    yticklabels=names_reordered,
    ax=ax, linewidths=0.3,
    vmin=-0.5, vmax=1.0,
    square=True,
)
ax.set_title('Emotion Similarity Based on Shared Neuron Selectivity Profiles\n'
             '(cosine similarity in PCA-reduced space, ordered by dendrogram)',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'neuron_emotion_similarity.png'),
            dpi=200, bbox_inches='tight')
plt.show()

## 7. Save Results

In [ ]:
# Save all results
import pickle

# 1. Neuron catalog (top neurons per emotion)
catalog_path = os.path.join(results_dir, 'neuron_catalog.csv')
df_catalog.to_csv(catalog_path, index=False)
print(f'Saved: {catalog_path} ({len(df_catalog)} rows)')

# 2. Significant neuron counts per emotion
sig_path = os.path.join(results_dir, 'neuron_significant_counts.csv')
df_sig.to_csv(sig_path, index=False)
print(f'Saved: {sig_path} ({len(df_sig)} rows)')

# 3. Selectivity matrices (per layer)
selectivity_path = os.path.join(results_dir, 'neuron_selectivity.pkl')
with open(selectivity_path, 'wb') as f:
    pickle.dump(selectivity, f)
print(f'Saved: {selectivity_path} ({n_layers} layers x {n_neurons} neurons x {NUM_LABELS} emotions)')

# 4. SVD-neuron connection analysis
svd_connection_path = os.path.join(results_dir, 'svd_neuron_connection.pkl')
with open(svd_connection_path, 'wb') as f:
    pickle.dump({
        'rank': ANALYSIS_RANK,
        'results': svd_neuron_results,
        'frac_removed_matrix': frac_removed_matrix,
        'mean_frac_removed': mean_frac_removed,
        'per_emotion_f1_drop': per_emotion_f1_drop,
    }, f)
print(f'Saved: {svd_connection_path}')

# 5. Cluster assignments
cluster_df = pd.DataFrame({
    'emotion': EMOTION_NAMES,
    'cluster': cluster_labels,
    'mean_frac_removed': mean_frac_removed,
    'total_sig_neurons': [df_sig[df_sig['emotion'] == e]['total_significant'].values[0]
                          for e in EMOTION_NAMES],
})
cluster_path = os.path.join(results_dir, 'neuron_emotion_clusters.csv')
cluster_df.to_csv(cluster_path, index=False)
print(f'Saved: {cluster_path}')

# ---------- Summary ----------
print('\n' + '=' * 70)
print('SUMMARY — Phase 7: FFN Neuron Specialization Analysis')
print('=' * 70)
print(f'\nDataset: {n_samples} test examples, {NUM_LABELS} emotions')
print(f'Neurons analyzed: {n_layers} layers x {n_neurons} neurons = {n_layers * n_neurons:,} total')
print(f'Selectivity threshold: |d| > {SELECTIVITY_THRESHOLD}')
print(f'SVD analysis rank: {ANALYSIS_RANK}')

total_sig = sum(
    (np.abs(selectivity[l]).max(axis=1) > SELECTIVITY_THRESHOLD).sum()
    for l in range(n_layers)
)
print(f'\nSpecialized neurons: {total_sig:,} / {n_layers * n_neurons:,} '
      f'({total_sig / (n_layers * n_neurons) * 100:.1f}%)')
print(f'Max selectivity observed: {np.abs(all_sel).max():.3f}')

print(f'\nSVD-neuron correlation: Spearman rho = {rho:.3f} (p = {p_value:.4f})')

print(f'\nFigures saved to results/:')
figures = [
    'neuron_selectivity_heatmap.png',
    'neuron_specialization_per_emotion.png',
    'neuron_selectivity_distributions.png',
    'svd_neuron_vulnerability_heatmap.png',
    'svd_vulnerability_vs_f1_drop.png',
    'neuron_emotion_dendrogram.png',
    'neuron_emotion_similarity.png',
]
for fig_name in figures:
    path = os.path.join(results_dir, fig_name)
    exists = 'ok' if os.path.exists(path) else 'MISSING'
    print(f'  [{exists}] {fig_name}')

print(f'\nData files:')
data_files = [
    'neuron_catalog.csv',
    'neuron_significant_counts.csv',
    'neuron_selectivity.pkl',
    'svd_neuron_connection.pkl',
    'neuron_emotion_clusters.csv',
]
for fname in data_files:
    path = os.path.join(results_dir, fname)
    exists = 'ok' if os.path.exists(path) else 'MISSING'
    print(f'  [{exists}] {fname}')